# Differential expression

In [ ]:
import os
import pickle as pkl

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

## QC on raw counts

Before we do any differential expression analysis, it's good to do some QC on the count data to ensure the sequencing samples cluster as expected. For our experiment, we expect 2 clusters (Cortisol vs. DMSO). There were 14 DMSO samples and 12 cortisol samples.

### Gene filtering based on total read count and the number of samples the gene is found in

First, we are going to filter the count data to remove genes with low total expression values (anything less than 10 is not going to be informative) and genes that were only expressed in 3 or fewer samples. First, we read in the counts data and convert numeric columns to integers:

In [ ]:
import pandas as pd

counts_df = pd.read_csv("/data/classes/2026/fall/biol343_001/_course_files/star_counts.tsv", sep="\t", header=1)
counts_df.iloc[:, 6:32] = counts_df.iloc[:, 6:32].astype('int')
counts_df

Next, we do the filtering. The first operation will sum the expression across all samples and remove any gene that had a total count of less than or equal to 10. The second operation will count the number of samples that had counts >0 of each gene. Any gene that wasn't expressed in >3 samples will be removed entirely.

In [ ]:
# take only the Geneid and count columns
star_cols = [c for c in counts_df.columns if "star.bam" in c]
counts_subset = counts_df[["Geneid"] + star_cols].copy()

# rename count columns
prefix = "/data/classes/2026/fall/biol343_001/_course_files/star.bam:"
def strip_prefix(name: str) -> str:
    return name.replace(prefix, "")
counts_subset.rename(columns={c: strip_prefix(c) for c in star_cols}, inplace=True)
counts_subset

Now we're going to sum the counts (row-wise). This will create the new column `total_counts`:

In [ ]:

# compute total_counts across numeric sample columns (row-wise)
# treat NA as 0, matching na.rm = TRUE
sample_cols = [c for c in counts_subset.columns if c != "Geneid"]
counts_numeric = counts_subset[sample_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
counts_subset["total_counts"] = counts_numeric.sum(axis=1)

counts_subset 

Filter to only keep genes with `total_counts` >=10:

In [ ]:
counts_summary = counts_subset.loc[counts_subset["total_counts"] >= 10].copy()
counts_summary

You can see that previously, we had 53,176 rows. Each row is a gene. Now, we have 37,561 rows. We removed 15,615 lowly expressed genes. 

Next, we'll find genes that were detected (count >0) in 6 or fewer samples. Even if their total count passed the first filter, these reads are concentrated in just a subset samples, so they aren't informative for comparing groups. We'll remove them.

In [ ]:
positive_counts_mask = counts_numeric.loc[counts_summary.index] > 0
n_positive_samples = positive_counts_mask.sum(axis=1)
genes_to_remove = counts_summary.loc[n_positive_samples <= 6, "Geneid"]
genes_to_remove

373 genes are going to be removed. Let's remove them:

In [ ]:
counts_filt = (
    counts_summary.loc[~counts_summary["Geneid"].isin(genes_to_remove)]
    .sort_values(by="Geneid")
    .drop(columns=["total_counts"])
)
counts_filt

This filtered dataset of 37188 genes will be used for all downstream analyses. 

### Clustering for QC
An easy QC plot is to cluster the samples based on the similarity of their gene-wise expression counts. Samples with the same treatment should cluster together.

We'll measure the Euclidean distance between each sample and then plot the results as a heatmap. We'll convert the data from a data frame to a matrix, and then measure the distance.

We'll first get the counts and create a matrix object and set the rownames as Geneid:

In [ ]:
import numpy as np

# get the names of the columns that have the counts
count_cols = [c for c in counts_filt.columns if c != "Geneid"]

# convert those coulumns to numeric
counts_m = counts_filt[count_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

# make the index of counts_m the Geneid column from the counts_filt
counts_m.index = counts_filt["Geneid"].values
counts_m

We'll use NumPy to calculate the distances. We have to transpose the matrix (samples x genes instead of genes x samples), then we'll calculate the distances:

In [ ]:
from sklearn.metrics import pairwise_distances

# transpose the matrix
X = counts_m.to_numpy().T  

# get a list of the sample names
sample_names = counts_m.columns.tolist()

# calculate the distance between samples
D = pairwise_distances(X, metric="euclidean")

# convert those distances to a data framee
dists_df = pd.DataFrame(D, index=sample_names, columns=sample_names)

dists_df.round(0)

You can see the distance matrix, which shows how similar in expression each sample is to every other samples. QC involves us ensuring that the Cortisol samples are all closer to each other than to the DMSO samples. You can see that by eye, but here's a heatmap showing that:

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.clustermap(
    dists_df,
    cmap="viridis",
    method="average",
    z_score=None,
    standard_scale=None,
    figsize=(8, 8)
)


With unnormalized counts, the similarities don't cluster all that well. We can also see that with a PCA.

We first log-transform the count data and scale the data across samples:

In [ ]:

from sklearn.preprocessing import StandardScaler

# log transform and transpose to samples x genes again
X = np.log10(counts_m.to_numpy() + 1.0).T 
sample_names = counts_m.columns.tolist()

# scale features (genes) across samples
scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X)
X_scaled

We're also going to build a DataFrame that includes sample metadata. This will be used to annotate the PCA and will also used for DESeq2 analyses later.

In [ ]:
sample_ids = list(counts_m.columns)

# create a DataFrame with sample_id
metadata = pd.DataFrame({"sample_id": sample_ids})
metadata
parts = metadata["sample_id"].str.split("_", expand=True)
parts = metadata["sample_id"].str.extract(
    r"^(?P<treatment>[^_]+)_(?P<bio_rep>\d+)(?P<tech_rep>[a-z]+)$"
)
parts

metadata = pd.concat([metadata, parts], axis=1)
metadata = metadata.set_index("sample_id")
metadata

Next we run the PCA and plot the points:

In [ ]:

from sklearn.decomposition import PCA

pca = PCA(n_components=10, random_state=0)  
scores = pca.fit_transform(X_scaled)        
# explained = pca.explained_variance_ratio_

scores_df = pd.DataFrame(
    scores[:, :2],  # PC1 and PC2 for plotting
    index=sample_names,
    columns=["PC1", "PC2"]
)

plot_df = scores_df.join(metadata[["treatment", "bio_rep"]], how="left")

# plot
plt.figure(figsize=(7, 6))
sns.scatterplot(
    data=plot_df,
    x="PC1",
    y="PC2",
    hue="treatment",     # color by treatment
    style="bio_rep",     # shape bi bio_rep
    s=80
)
plt.title("PCA: PC1 vs PC2")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0.)
plt.tight_layout()
plt.show()


A few things stand out here: 
1. We don't see grouping by treatment, just like we didn't in the clustering. Why not?
2. Technical replicates (same shape and same treatment) cluster in pairs, but pairs are far away from each other.

#1 is a problem. If the treatment doesn't clearly discriminate transcriptomes, can we really trust any DEGs that we find?

Let's see if normalizing and transforming counts with DESeq2 can solve our clustering problem.

## Differential expression analysis
To perform differential expression analysis, DESeq2 requires two types of input:

1. A count matrix of shape ‘number of samples’ x ‘number of genes’, containing read counts (non-negative integers) 
2. Metadata (or “column” data) of shape ‘number of samples’ x ‘number of variables’, containing sample annotations that will be used to split the data in cohorts.

The output of featureCounts needs to be converted to a matrix, the column names should be simplified, and a few unnecessary columns need to be removed. We also made the metadata DataFrame earlier as well.

The 26 libraries come from 6 biological replicates (3 DMSO, 3 cortisol), each sequenced as 4–6 technical replicates. DESeq2 assumes every column is an independent biological sample, so technical replicates would inflate the sample size and produce far too many DEGs. Following DESeq2's recommendation, we sum the raw counts of technical replicates within each biological replicate.

In [ ]:
# biological replicate ID for each library, e.g. DMSO_1, cortisol_3
bio_id = metadata["treatment"].astype(str) + "_" + metadata["bio_rep"].astype(str)

# sum raw counts across technical replicates (genes x biological replicates)
counts_bio = counts_m.T.groupby(bio_id).sum().T.astype(int)

# one metadata row per biological replicate
metadata_bio = metadata.groupby(bio_id)[["treatment", "bio_rep"]].first()
metadata_bio["n_tech_reps"] = metadata.groupby(bio_id).size()
metadata_bio.index.name = "sample_id"

metadata_bio


To be sure, we'll check to confirm the metadata and count columns from `counts_m` correspond:

In [ ]:
set(metadata_bio.index) == set(counts_bio.columns)

### Single factor analyses

DESeq2 allows for single factorial analyses (only one independent variable, i.e., just treatment) or multifactorial analyses (more than independent variable, which is not applicaable for this dataset). We will analyze the data to see if there are differences between DMSO and Cortisol-treated fish. To do so, we create a `DeseqDataSet` (or DDS object), which incorporates the counts and the metadata. We can then run the `deseq()` method on the DDS object to fit dispersions and log-fold changes:

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats  

inference = DefaultInference(n_cpus=32)

dds = DeseqDataSet(
    counts=counts_bio.T,         # samples x genes, integer counts
    metadata=metadata_bio,       # DataFrame indexed by sample_id
    design_factors="treatment",
    refit_cooks=True,
    inference=inference
)

dds


The DDS object is based on the AnnData object. Like any Python objects, we can access the fields:

In [ ]:
print(dds.obsm)
dds.obsm['design_matrix']

Now we can run the `deseq2()` method, which fits the dispersion and log-fold changes and therefore now adds new fields to the DDS object:

In [ ]:
dds.deseq2()
dds

Now, for example, we can access the gene-level log-fold changes (LFCs):

In [ ]:
dds.varm["LFC"]

The DDS object with the dispersions and LFCs allow us to perform statistical tests. The `DeseqStats` class includes the DDS object and will allow for the calculation of p-values and adjusted p-values. These data are stored in `results_df`. We can view these data with the `.summary()` methods

In [ ]:
stat_res = DeseqStats(dds, inference=inference)
stat_res.summary()

One of the featured differentially expressed genes from the paper was the irg1l gene, which is involved in immunity. Fig. 2e showed that this gene decreased in expression after cortisol treatment. We can search for that gene to see we reproduce the finding:

In [ ]:
summary = stat_res.results_df
summary['gene_id'] = summary.index
summary = summary.reset_index()
summary.loc[summary['gene_id'] == 'irg1l']


We can see that the L2FC of irg1l is -2.34, which means its expression is 2^-2.34 (~0.19) in treated samples as in control samples. This is consistent with the figure from the paper:

<img src="../8_counting/assets/m_bio02006502.jpeg" width="400">

### Volcano plots

Typically in RNA-seq analyses, we are interested in genes that are significantly differentially expressed (padj <0.05) and those that have large differences (maybe L2FC > 1, which is a two-fold change). Volcano plots are a good way to visualize all the genes that satisfy both conditions.

In [ ]:
x = summary['log2FoldChange'].to_numpy()
y = -np.log10(summary['padj'].to_numpy())

sig = (summary['padj'] < 0.05) & (summary['log2FoldChange'].abs() >= 1)
plt.figure(figsize=(6, 4))
plt.axhline(-np.log10(0.05), color='red', linestyle='--', linewidth=1)
plt.axvline(1, color='gray', linestyle='--', linewidth=1)
plt.axvline(-1, color='gray', linestyle='--', linewidth=1)
plt.scatter(x[~sig], y[~sig], s=3, color='steelblue', alpha=0.7, label='NS')
plt.scatter(x[sig], y[sig], s=6, color='tomato', alpha=0.8, label='DE (padj<0.05 & |LFC|≥2)')
plt.legend(frameon=False)
plt.xlabel('log2FoldChange'); plt.ylabel('-log10(padj)')
plt.title('Volcano plot'); plt.tight_layout(); plt.show()


In a plot like this, the higher points represent genes that are most likely to be truly significantly differentially expressed (a small p-value). The points far to the right or left represent genes that have large expression differences (fold changes) between DMSO and Cortisol. The points in the top right/left areas (red), then, are the genes in which we're interested.

In [ ]:
# keep only significant DEGs, with gene ID, log2FC, and adjusted p-value
degs = (
    summary.loc[sig, ['gene_id', 'log2FoldChange', 'padj']]
    .sort_values('padj')
)

degs.to_csv('/data/classes/2026/fall/biol343_001/_course_files/DEGs_cortisol_vs_DMSO.csv', index=False)


## QC on transformed data

Now what we have the `dds` object, we can do another PCA on counts that have been normalized and transformed. The first PCA we did was on raw counts, but it's often helpful to see the PCA clusters on normalized/transformed counts. We first transform the read counts with DESeq2's variance-stabilizing transformation, which will add a new `vst_counts` layer to the `dds` object:

In [ ]:
dds.vst()


Now we run the same PCA code, but this time using the normalized/transformed counts and collapsing the technical replicates:

In [ ]:
X = dds.layers["vst_counts"]   # shape: (samples, genes)
obs = dds.obs  # sample metadata DataFrame

pca = PCA(n_components=5, random_state=0)   # can't exceed 6 samples

scores = pca.fit_transform(X)        
# explained = pca.explained_variance_ratio_

scores_df = pd.DataFrame(
    scores[:, :2],  # PC1 and PC2 for plotting
    index=dds.obs_names,
    columns=["PC1", "PC2"]
)

plot_df = scores_df.join(metadata_bio[["treatment", "bio_rep"]], how="left")

# plot
plt.figure(figsize=(7, 6))
sns.scatterplot(
    data=plot_df,
    x="PC1",
    y="PC2",
    hue="treatment",     # color by tissue
    style="bio_rep", 
    s=80
)
plt.title("PCA: PC1 vs PC2")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0.)
plt.tight_layout()
plt.show()



With collapsed techincal replicates and normalized/transformed data, we see two clear clusters.